# PIB per Capita dos Municípios Brasileiros — Extração direta do IBGE

Todas as tabelas são extraídas via `sidrapy` diretamente da API do SIDRA/IBGE.

| Dado | Tabela SIDRA | Variável | Unidade |
|---|---|---|---|
| PIB total | 5938 | 37 | Mil Reais |
| VAB Agropecuária | 5938 | 513 | Mil Reais |
| VAB Indústria | 5938 | 517 | Mil Reais |
| VAB Serviços | 5938 | 6575 | Mil Reais |
| VAB Adm. Pública | 5938 | 525 | Mil Reais |
| Part. Agropecuária | 5938 | 516 | % |
| Part. Indústria | 5938 | 520 | % |
| Part. Serviços | 5938 | 6574 | % |
| Part. Adm. Pública | 5938 | 528 | % |
| População estimada | 6579 | 9324 | Pessoas |
| População Censo 2022 | 9514 | 93 | Pessoas |

**Período:** 2003–2023  
**Anos 2007 e 2010:** população ausente na tabela 6579 → interpolação linear entre anos adjacentes.

## 1. Instalação e imports

In [13]:
import sidrapy
import pandas as pd
import numpy as np
import time
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)

DELAY = 1.5  # segundos entre chamadas à API
NULOS = ['...', '-', 'X', 'x', '..', '', ' ']



## 2. Funções auxiliares

In [14]:
def extrair_sidra(table_code, variable, anos, label=''):
    """Extrai uma variável da tabela 5938 para todos os municípios, ano a ano."""
    frames = []
    erros = []
    for ano in anos:
        try:
            df = sidrapy.get_table(
                table_code=table_code,
                territorial_level='6',
                ibge_territorial_code='all',
                variable=variable,
                period=str(ano),
                header='y'
            ).iloc[1:].copy()
            frames.append(df)
            print(f'  ✔ {label} {ano} | {len(df):,} municípios')
        except Exception as e:
            erros.append(ano)
            print(f'  ✘ {label} {ano} | ERRO: {e}')
        time.sleep(DELAY)
    if erros:
        print(f'  Anos com erro: {erros}')
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def limpar_valor(df_raw, col_valor, novo_nome, fator=1):
    """Padroniza um DataFrame bruto do SIDRA."""
    df = df_raw.rename(columns={
        'D1C': 'cod_municipio', 'D1N': 'nome_municipio',
        'D2C': 'ano', 'V': col_valor
    }).copy()
    df['cod_municipio'] = df['cod_municipio'].astype(str).str.strip().str.zfill(7)
    df['ano'] = pd.to_numeric(df['ano'], errors='coerce').astype('Int64')
    df[col_valor] = (
        df[col_valor].replace(NULOS, pd.NA)
        .astype(str).str.replace(',', '.', regex=False)
        .pipe(pd.to_numeric, errors='coerce')
    ) * fator
    return (
        df.loc[df['cod_municipio'].str.match(r'^\d{7}$')]
        .dropna(subset=['ano'])
        .rename(columns={col_valor: novo_nome})
        [['cod_municipio', 'nome_municipio', 'ano', novo_nome]]
        .sort_values(['cod_municipio', 'ano'])
        .reset_index(drop=True)
    )

print('Funções definidas.')

Funções definidas.


## 3. Extração do PIB e VAB setorial (tabela 5938)

Todas as variáveis monetárias estão em **Mil Reais** na API.

In [15]:
ANOS = list(range(2010, 2024))  # 2003 a 2023


In [16]:
ANOS = list(range(2010, 2024))  # 2003 a 2023

# Mapa: nome_coluna → (variável SIDRA, fator de conversão)
# fator 1000 = Mil Reais → Reais
# fator 1    = percentual (já em %)
VARIAVEIS_5938 = {
    'pib_reais'           : ('37',   1000),
    'vab_agro_reais'      : ('513',  1000),
    'vab_ind_reais'       : ('517',  1000),
    'vab_serv_reais'      : ('6575', 1000),
    'vab_adm_reais'       : ('525',  1000),
    'part_agropecuaria'   : ('516',  1),
    'part_industria'      : ('520',  1),
    'part_servicos'       : ('6574', 1),
    'part_adm_publica'    : ('528',  1),
}

dfs_setorial = {}

for nome_col, (var_id, fator) in VARIAVEIS_5938.items():
    print(f'\nExtraindo: {nome_col} (variável {var_id})')
    raw = extrair_sidra('5938', var_id, ANOS, label=nome_col)
    dfs_setorial[nome_col] = limpar_valor(raw, 'V', nome_col, fator=fator)

print('\n✔ Extração da tabela 5938 concluída.')

  ✔ part_servicos 2015 | 5,570 municípios
  ✔ part_servicos 2016 | 5,570 municípios
  ✔ part_servicos 2017 | 5,570 municípios
  ✔ part_servicos 2018 | 5,570 municípios
  ✔ part_servicos 2019 | 5,570 municípios
  ✔ part_servicos 2020 | 5,570 municípios
  ✔ part_servicos 2021 | 5,570 municípios
  ✔ part_servicos 2022 | 5,570 municípios
  ✔ part_servicos 2023 | 5,570 municípios

Extraindo: part_adm_publica (variável 528)
  ✔ part_adm_publica 2010 | 5,570 municípios
  ✔ part_adm_publica 2011 | 5,570 municípios
  ✔ part_adm_publica 2012 | 5,570 municípios
  ✔ part_adm_publica 2013 | 5,570 municípios
  ✔ part_adm_publica 2014 | 5,570 municípios
  ✔ part_adm_publica 2015 | 5,570 municípios
  ✔ part_adm_publica 2016 | 5,570 municípios
  ✔ part_adm_publica 2017 | 5,570 municípios
  ✔ part_adm_publica 2018 | 5,570 municípios
  ✔ part_adm_publica 2019 | 5,570 municípios
  ✔ part_adm_publica 2020 | 5,570 municípios
  ✔ part_adm_publica 2021 | 5,570 municípios
  ✔ part_adm_publica 2022 | 5,570 muni

In [17]:
# Consolida todas as variáveis da 5938 em um único DataFrame
df_pib = dfs_setorial['pib_reais'][['cod_municipio', 'nome_municipio', 'ano', 'pib_reais']].copy()

for nome_col, df_var in dfs_setorial.items():
    if nome_col == 'pib_reais':
        continue
    df_pib = df_pib.merge(
        df_var[['cod_municipio', 'ano', nome_col]],
        on=['cod_municipio', 'ano'],
        how='left'
    )

# Extrai UF do código IBGE (2 primeiros dígitos)
UF_MAP = {
    '11':'RO','12':'AC','13':'AM','14':'RR','15':'PA','16':'AP','17':'TO',
    '21':'MA','22':'PI','23':'CE','24':'RN','25':'PB','26':'PE','27':'AL',
    '28':'SE','29':'BA','31':'MG','32':'ES','33':'RJ','35':'SP','41':'PR',
    '42':'SC','43':'RS','50':'MS','51':'MT','52':'GO','53':'DF'
}
df_pib['sigla_uf'] = df_pib['cod_municipio'].str[:2].map(UF_MAP)

print(f'PIB consolidado: {df_pib["cod_municipio"].nunique():,} municípios × {df_pib["ano"].nunique()} anos')
print(f'Colunas: {df_pib.columns.tolist()}')

PIB consolidado: 5,570 municípios × 14 anos
Colunas: ['cod_municipio', 'nome_municipio', 'ano', 'pib_reais', 'vab_agro_reais', 'vab_ind_reais', 'vab_serv_reais', 'vab_adm_reais', 'part_agropecuaria', 'part_industria', 'part_servicos', 'part_adm_publica', 'sigla_uf']


## 4. Extração da população

- **2010–2021:** tabela `6579`, variável `9324` (estimativas enviadas ao TCU via SIDRA)
- **2022:** tabela `9514`, variável `93` (Censo Demográfico 2022)
- **2023:** arquivo XLS do FTP do IBGE — `POP_TCU_2023_Municipios_POP2022_Malha2023.xls`
- **2010:** ausente na tabela 6579 → interpolação linear entre 2009 e 2011

In [43]:
import requests
import io

# ── 4a. Estimativas TCU via SIDRA (2011–2021) ────────────────────────────────
ANOS_TCU = [a for a in ANOS if a not in (2022, 2023, 2024)]
print('Extraindo população (tabela 6579)...')
raw_pop_tcu = extrair_sidra('6579', '9324', ANOS_TCU, label='Pop TCU')

time.sleep(DELAY)

# ── 4b. Censo 2022 (tabela 9514) ─────────────────────────────────────────────
print('\nExtraindo Censo 2022 (tabela 9514)...')
raw_pop_censo = extrair_sidra('9514', '93', [2022], label='Censo 2022')

print('\n✔ Extração SIDRA de população concluída.')


# ── 4c. Função para baixar estimativas do FTP do IBGE (XLS) ──────────────────
def extrair_pop_xls_ibge(url_xls, ano):
    """
    Estrutura do arquivo (ex: POP_TCU_2023):
      - Linha 0: título descritivo  → ignorada via skiprows=1
      - Linha 1: cabeçalho:
          'COD. UF'      → 2 dígitos da UF
          'COD. MUNIC'   → 5 dígitos do município
          'NOME DO MUNICÍPIO'
          coluna de população

    Código IBGE completo (7d) = COD_UF(2d) + COD_MUNIC(5d)
    """
    headers = {'User-Agent': 'Mozilla/5.0'}
    resp = requests.get(url_xls, timeout=120, headers=headers)
    resp.raise_for_status()

    df_raw = pd.read_excel(
        io.BytesIO(resp.content),
        engine='xlrd',
        skiprows=1,
        dtype=str
    )
    df_raw.columns = df_raw.columns.str.strip()
    cols_upper = {c: c.upper() for c in df_raw.columns}

    col_uf   = next(c for c, u in cols_upper.items()
                    if 'COD' in u and 'UF' in u and 'MUNIC' not in u)
    col_mun  = next(c for c, u in cols_upper.items()
                    if 'COD' in u and 'MUNIC' in u)
    col_nome = next(c for c, u in cols_upper.items()
                    if 'NOME' in u and 'MUNIC' in u)
    col_pop  = next(c for c, u in cols_upper.items()
                    if 'POPUL' in u or 'APURADA' in u or 'ESTIMAT' in u or 'HABITANT' in u)

    df = df_raw[[col_uf, col_mun, col_nome, col_pop]].copy()
    df.columns = ['cod_uf', 'cod_mun', 'nome_municipio', 'populacao']

    # Reconstrói código IBGE de 7 dígitos: COD_UF(2d) + COD_MUNIC(5d)
    df['cod_uf']  = (df['cod_uf'].astype(str).str.strip()
                     .str.replace(r'\.0$', '', regex=True).str.zfill(2))
    df['cod_mun'] = (df['cod_mun'].astype(str).str.strip()
                     .str.replace(r'\.0$', '', regex=True).str.zfill(5))
    df['cod_municipio'] = df['cod_uf'] + df['cod_mun']

    df['populacao'] = pd.to_numeric(
        df['populacao'].astype(str)
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False),
        errors='coerce'
    )
    df['ano'] = ano

    df = (
        df.loc[df['cod_municipio'].str.match(r'^\d{7}$')]
        .dropna(subset=['populacao'])
        .reset_index(drop=True)
    )
    print(f'  ✔ Pop {ano} via XLS FTP IBGE | {len(df):,} municípios')
    return df[['cod_municipio', 'nome_municipio', 'ano', 'populacao']]


# ── 4d. Download XLS 2023 do FTP do IBGE ─────────────────────────────────────
BASE_FTP = 'https://ftp.ibge.gov.br'

URL_POP_2023 = (
    BASE_FTP
    + '/Informacoes_Gerais_e_Referencia'
    + '/Relacao_da_Populacao_dos_Municipios_para_publicacao_no_DOU_em_2023'
    + '/POP_TCU_2023_Municipios_POP2022_Malha2023.xls'
)

print('\nBaixando estimativas 2023 do FTP do IBGE...')
df_pop_2023 = extrair_pop_xls_ibge(URL_POP_2023, ano=2023)

print('\n✔ Download das estimativas XLS concluído.')

Extraindo população (tabela 6579)...
  ✔ Pop TCU 2011 | 5,571 municípios
  ✔ Pop TCU 2012 | 5,571 municípios
  ✔ Pop TCU 2013 | 5,571 municípios
  ✔ Pop TCU 2014 | 5,571 municípios
  ✔ Pop TCU 2015 | 5,571 municípios
  ✔ Pop TCU 2016 | 5,571 municípios
  ✔ Pop TCU 2017 | 5,571 municípios
  ✔ Pop TCU 2018 | 5,571 municípios
  ✔ Pop TCU 2019 | 5,571 municípios
  ✔ Pop TCU 2020 | 5,571 municípios
  ✔ Pop TCU 2021 | 5,571 municípios

Extraindo Censo 2022 (tabela 9514)...
  ✔ Censo 2022 2022 | 5,570 municípios

✔ Extração SIDRA de população concluída.

Baixando estimativas 2023 do FTP do IBGE...
  ✔ Pop 2023 via XLS FTP IBGE | 5,541 municípios

✔ Download das estimativas XLS concluído.


In [44]:
# Limpa as fontes SIDRA
df_pop_tcu   = limpar_valor(raw_pop_tcu,   'V', 'populacao', fator=1)
df_pop_censo = limpar_valor(raw_pop_censo, 'V', 'populacao', fator=1)
df_pop_censo['ano'] = 2022

# Consolida: SIDRA (2011–2021) + Censo 2022 + XLS 2023
df_pop = (
    pd.concat([df_pop_tcu, df_pop_censo, df_pop_2023], ignore_index=True)
    .drop_duplicates(['cod_municipio', 'ano'])
    .sort_values(['cod_municipio', 'ano'])
    .reset_index(drop=True)
)

ANOS = sorted(df_pop['ano'].unique().tolist())

anos_com_pop = sorted(df_pop['ano'].unique())
anos_sem_pop = [a for a in ANOS if a not in anos_com_pop]
print(f'Anos com população : {anos_com_pop}')
print(f'Anos SEM população (serão interpolados): {anos_sem_pop}')

Anos com população : [np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Anos SEM população (serão interpolados): []


## 5. Cálculo do PIB per capita

In [45]:
df = df_pib.merge(
    df_pop[['cod_municipio', 'ano', 'populacao']],
    on=['cod_municipio', 'ano'],
    how='left'
)

df['pib_percapita_rs'] = df['pib_reais'] / df['populacao']

print(f'Observações totais          : {len(df):,}')
print(f'Com PIB per capita calculado: {df["pib_percapita_rs"].notna().sum():,}')
print(f'Sem população               : {df["populacao"].isna().sum():,}')

Observações totais          : 77,980
Com PIB per capita calculado: 72,371
Sem população               : 5,604


## 6. Validação — Florianópolis

| Ano | Referência IBGE Cidades |
|---|---|
| 2021 | R\$ 45.602,98 |
| 2023 | R\$ 58.059,37 |

In [46]:
floripa = df[df['cod_municipio'] == '4205407'].sort_values('ano')

print('Florianópolis — série histórica:')
print(floripa[['ano', 'pib_reais', 'populacao', 'pib_percapita_rs']].to_string(index=False))

print()
refs = {2021: 45602.98, 2023: 58059.37}
for ano_r, val_r in refs.items():
    s = floripa.loc[floripa['ano'] == ano_r, 'pib_percapita_rs']
    if not s.empty and pd.notna(s.values[0]):
        diff = abs(s.values[0] - val_r)
        ok = '✔ OK' if diff < 10 else f'⚠ Diferença: R$ {diff:,.2f}'
        print(f'{ano_r}: calculado R$ {s.values[0]:,.2f} | referência R$ {val_r:,.2f} | {ok}')
    else:
        print(f'{ano_r}: não disponível')

Florianópolis — série histórica:
 ano         pib_reais  populacao  pib_percapita_rs
2010 11,276,680,000.00        NaN               NaN
2011 12,731,618,000.00 427,298.00         29,795.64
2012 13,946,621,000.00 433,158.00         32,197.54
2013 14,974,993,000.00 453,285.00         33,036.60
2014 16,915,926,000.00 461,524.00         36,652.32
2015 17,619,984,000.00 469,690.00         37,514.07
2016 18,660,876,000.00 477,798.00         39,055.99
2017 19,516,694,000.00 485,838.00         40,171.20
2018 21,059,944,000.00 492,977.00         42,719.93
2019 21,966,053,000.00 500,973.00         43,846.78
2020 21,300,109,000.00 508,826.00         41,861.28
2021 23,559,947,000.00 516,524.00         45,612.49
2022 26,552,426,000.00 537,211.00         49,426.44
2023 31,190,132,000.00 537,211.00         58,059.37

2021: calculado R$ 45,612.49 | referência R$ 45,602.98 | ✔ OK
2023: calculado R$ 58,059.37 | referência R$ 58,059.37 | ✔ OK


## 7. Análise exploratória

In [47]:
# Evolução nacional
evolucao = (
    df.groupby('ano')['pib_percapita_rs']
    .agg(n_municipios='count', mediana='median', media='mean', minimo='min', maximo='max')
    .reset_index()
)
print('Evolução nacional — PIB per capita municipal (R$):')
display(evolucao)

Evolução nacional — PIB per capita municipal (R$):


,ano,n_municipios,mediana,media,minimo,maximo
0,2010,0,NaN,NaN,NaN,NaN
1,2011,5565,"10,580.19","14,435.34","2,574.49","622,975.90"
2,2012,5565,"11,555.38","15,798.48","-1,459.80","777,099.82"
3,2013,5570,"12,711.63","17,390.68",301.61,"717,343.67"
4,2014,5570,"13,879.95","18,674.34","3,081.72","815,697.80"
5,2015,5570,"14,610.61","19,635.49","3,089.54","513,142.01"
6,2016,5570,"15,860.63","20,994.00","3,191.18","364,529.33"
7,2017,5570,"16,613.33","22,042.72","3,289.52","346,739.34"
8,2018,5570,"17,413.10","23,482.85","4,778.49","582,654.68"
9,2019,5570,"18,224.49","24,632.70","4,475.19","465,097.46"


In [48]:
# Top 10 no ano mais recente
ano_max = int(df['ano'].max())
top10 = (
    df[df['ano'] == ano_max]
    .nlargest(10, 'pib_percapita_rs')
    [['cod_municipio', 'nome_municipio', 'sigla_uf', 'pib_percapita_rs']]
)
print(f'Top 10 municípios — PIB per capita {ano_max} (R$):')
display(top10)

Top 10 municípios — PIB per capita 2023 (R$):


,cod_municipio,nome_municipio,sigla_uf,pib_percapita_rs
45583,3305505,Saquarema - RJ,RJ,"722,441.52"
30575,2929206,São Francisco do Conde - BA,BA,"684,319.24"
45023,3302700,Maricá - RJ,RJ,"679,714.48"
51491,3536505,Paulínia - SP,SP,"606,740.73"
44183,3204302,Presidente Kennedy - ES,ES,"537,982.70"
48999,3520400,Ilhabela - SP,SP,"424,535.27"
74199,5107768,Santa Rita do Trivelato - MT,MT,"409,443.53"
50063,3527306,Louveira - SP,SP,"388,732.46"
45485,3305000,São João da Barra - RJ,RJ,"382,417.41"
35335,3125101,Extrema - MG,MG,"377,790.64"


In [49]:
# Mediana por UF no último ano
por_uf = (
    df[df['ano'] == ano_max]
    .groupby('sigla_uf')['pib_percapita_rs']
    .agg(n_municipios='count', mediana='median', media='mean')
    .sort_values('mediana', ascending=False)
    .reset_index()
)
print(f'PIB per capita mediano por UF — {ano_max}:')
display(por_uf)

PIB per capita mediano por UF — 2023:


,sigla_uf,n_municipios,mediana,media
0,DF,1,"129,790.44","129,790.44"
1,MS,79,"67,060.19","78,124.98"
2,MT,141,"59,335.91","86,769.91"
3,SC,295,"52,591.22","58,022.05"
4,RS,497,"47,185.14","54,192.57"
5,SP,645,"46,614.00","57,736.45"
6,PR,399,"45,877.10","51,039.87"
7,RO,51,"42,868.54","47,793.94"
8,GO,246,"38,863.99","51,355.64"
9,RJ,92,"37,042.01","73,650.92"


## 8. Exportação

In [51]:
COLUNAS = [
    'cod_municipio', 'nome_municipio', 'sigla_uf', 'ano',
    'pib_reais', 'populacao', 'pib_percapita_rs',
    'vab_agro_reais', 'vab_ind_reais', 'vab_serv_reais', 'vab_adm_reais',
    'part_agropecuaria', 'part_industria', 'part_servicos', 'part_adm_publica'
]
df_export = df[[c for c in COLUNAS if c in df.columns]].sort_values(['cod_municipio','ano']).reset_index(drop=True)

OUTPUT_CSV     = 'pib_percapita_municipios.csv'
OUTPUT_XLSX    = 'pib_percapita_municipios.xlsx'
OUTPUT_PARQUET = 'pib_percapita_municipios.parquet'

df_export.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig', sep=';', decimal=',')
print(f'Salvo: {OUTPUT_CSV}')

df_wide = df_export.pivot_table(
    index=['cod_municipio', 'nome_municipio', 'sigla_uf'],
    columns='ano', values='pib_percapita_rs'
).reset_index()
df_wide.columns.name = None

with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
    df_export.to_excel(writer, sheet_name='Longo', index=False)
    df_wide.to_excel(writer, sheet_name='Wide - PIB per capita', index=False)

print(f'Salvo: {OUTPUT_XLSX}')

# Exportação Parquet — formato colunar eficiente para análises posteriores
df_export.to_parquet(OUTPUT_PARQUET, index=False, engine='pyarrow', compression='snappy')
print(f'Salvo: {OUTPUT_PARQUET}')

print(f'\nResumo: {df_export["cod_municipio"].nunique():,} municípios × {df_export["ano"].nunique()} anos = {len(df_export):,} observações')


Salvo: pib_percapita_municipios.csv
Salvo: pib_percapita_municipios.xlsx
Salvo: pib_percapita_municipios.parquet

Resumo: 5,570 municípios × 14 anos = 77,980 observações
